In [4]:
# pip install --upgrade transformers

In [5]:
# pip install torch

In [8]:
# pip install accelerate

In [1]:
# Cek instalasi PyTorch dan GPU
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"  Total memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Used memory: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
        print(f"  Free memory: {(props.total_memory - torch.cuda.memory_allocated(i)) / 1024**3:.2f} GB")
else:
    print("Running on CPU")
    
# Cek transformers
import transformers
print(f"\nTransformers version: {transformers.__version__}")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU count: 1
GPU 0: NVIDIA B200
  Total memory: 178.35 GB
  Used memory: 0.00 GB
  Free memory: 178.35 GB


/home/kel7/notebooks/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Transformers version: 5.6.2


In [2]:
import torch
print(torch.cuda.device_count())

1


cek limit 


In [3]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("Total VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1024**3)

GPU: NVIDIA B200
Total VRAM (GB): 178.3533935546875


In [4]:
import torch

print("Allocated:", torch.cuda.memory_allocated(0) / 1024**3, "GB")
print("Reserved :", torch.cuda.memory_reserved(0) / 1024**3, "GB")

Allocated: 0.0 GB
Reserved : 0.0 GB


batas maksimal


In [5]:
# import torch

# try:
#     x = torch.zeros((100000, 100000), device="cuda")  # coba besarin
#     print("Berhasil allocate besar")
# except RuntimeError as e:
#     print("Gagal:", e)

In [ ]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"dtype": torch.bfloat16}, 
    device_map="auto",
)

c:\Program Files\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [7]:
messages = [
    {"role": "system", "content": "kamu adalah asisten yang membantu menjawab pertanyaan dengan bahasa yang mudah dipahami."},
    {"role": "user", "content": "siapa itu jusuf kalla ?"},
]

outputs = pipeline(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'Jusuf Kalla adalah seorang politikus Indonesia yang pernah menjabat sebagai Wakil Presiden Republik Indonesia pada periode 2004-2009 dan 2009-2014. Ia adalah tokoh penting dalam Partai Golkar dan telah menjabat sebagai Gubernur Sulawesi Selatan dari tahun 1989 hingga 1998.\n\nJusuf Kalla lahir pada tanggal 15 Mei 1942 di Jeneponto, Sulawesi Selatan. Ia memiliki latar belakang pendidikan yang solid, dengan menempuh pendidikan di Fakultas Ekonomi Universitas Hasanuddin dan Fakultas Hukum Universitas Gadjah Mada.\n\nSelama menjabat sebagai Wakil Presiden, Jusuf Kalla dikenal sebagai tokoh yang kuat dan berpengaruh dalam politik Indonesia. Ia juga dikenal sebagai tokoh yang berkomitmen untuk meningkatkan kesejahteraan rakyat dan memajukan perekonomian Indonesia.\n\nSetelah mengundurkan diri dari jabatan Wakil Presiden pada tahun 2014, Jusuf Kalla kembali ke dunia'}


# Fine-Tuning Llama 3.1 8B Instruct

Berikut adalah langkah-langkah untuk fine-tuning model Llama menggunakan QLoRA (Quantized Low-Rank Adaptation) untuk efisiensi memory.

In [ ]:
# pip install transformers accelerate peft bitsandbytes datasets trl unsloth

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Cell 2: Impor library dan muat model Llama 3.1 8B (4-bit QLoRA)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Bisa disesuaikan (RoPE scaling otomatis)
dtype = None           # Auto-deteksi (Float16 untuk GPU modern)
load_in_4bit = True    # Gunakan kuantisasi 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct",  # Ganti jika Anda punya path lokal: "path/ke/model"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    device_map = "auto", 
    # token = "hf_...",  # Jika model gated, masukkan token Hugging Face
)

# Terapkan PEFT (LoRA) pada model yang sudah terkuantisasi
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # Rank LoRA (jurnal pakai nilai kecil, bisa 8-64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],  # Semua linear layer
    lora_alpha = 16,
    lora_dropout = 0,     # Dropout 0 untuk performa optimal
    bias = "none",        # Tidak melatih bias
    use_gradient_checkpointing = "unsloth",  # Untuk menghemat memori
    random_state = 3407,
    use_rslora = False,   # Rank-stabilized LoRA opsional
    loftq_config = None,  # Tidak pakai LoftQ
)
print("Model siap dengan LoRA adaptor.")

==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.353 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 10.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model siap dengan LoRA adaptor.


In [4]:
# Cell 3: Buat dataset instruksi (format Alpaca)
from datasets import Dataset

# Contoh data – GANTI dengan file Anda sendiri
# data = [
#     {"instruction": "Jelaskan Pasal 21 Konstitusi India.", "output": "Pasal 21 menjamin perlindungan kehidupan dan kebebasan pribadi..."},
#     {"instruction": "Apa perbedaan antara IPC seksi 299 dan 300?", "output": "Seksi 299 mengatur tentang pembunuhan yang dapat dihukum mati..."},
#     # ... tambahkan ribuan pasangan
# ]

data = [
    {"instruction": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintah Pusat adalah siapa?", "output": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintah Pusat adalah Presiden Republik Indonesia yang memegang kekuasaan pemerintahan negara Republik Indonesia yang dibantu oleh Wakil Presiden dan menteri sebagaimana dimaksud dalam Undang-Undang Dasar Negara Republik Indonesia Tahun 1945."},
    {"instruction": "Dalam Pasal 1 Desa Biru, apa yang dimaksud dengan Daerah?", "output": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Daerah adalah Kabupaten Bandung."},
    {"instruction": "Menurut Pasal 1 Desa Biru, apa yang dimaksud dengan Pemerintahan Daerah?", "output": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintahan Daerah adalah penyelenggaraan urusan pemerintahan oleh pemerintah daerah dan dewan perwakilan rakyat daerah menurut asas otonomi dan tugas pembantuan dengan prinsip otonomi seluas-luasnya dalam sistem dan prinsip Negara Kesatuan Republik Indonesia sebagaimana dimaksud dalam Undang-Undang Dasa Negara Republik Indonesia Tahun 1945."},
    {"instruction": "Dalam Pasal 1 Desa Biru, siapa yang dimaksud dengan Pemerintah Daerah?", "output": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintah Daerah adalah kepala daerah sebagai unsur penyelenggara Pemerintahan Daerah yang memimpin pelaksanaan urusan pemerintahan yang menjadi kewenangan daerah otonom."},
    {"instruction": "Menurut Pasal 1 Desa Biru, apa yang dimaksud dengan Kewenangan Desa Biru?", "output": "Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Kewenangan Desa Biru adalah kewenangan yang dimiliki Desa Biru meliputi kewenangan dibidang penyelenggaraan Pemerintahan Desa Biru, pelaksanaan Pembangunan Desa Biru, Pembinaan Kemasyarakatan Desa Biru, dan Pemberdayaan Masyarakat Desa Biru berdasarkan prakarsa masyarakat, hak asal usul, dan adat istiadat Desa Biru."},
    {"instruction": "Berdasarkan Pasal 2 Desa Biru, apa yang dimaksud dengan daftar kewenangan desa Biru yang diatur dalam Peraturan Desa ini?", "output": "Berdasarkan Pasal 2 Desa Biru, daftar kewenangan desa Biru yang diatur dalam Peraturan Desa ini meliputi kewenangan berdasarkan hak asal usul dan kewenangan lokal berskala desa."},
    {"instruction": "Berdasarkan Pasal 2 Desa Biru, apa yang dimaksud dengan kewenangan berdasarkan hak asal usul?", "output": "Berdasarkan Pasal 2 Desa Biru, yang dimaksud dengan kewenangan berdasarkan hak asal usul adalah kewenangan yang dipegang oleh desa Biru berdasarkan peraturan yang mengatur hak asal usul tersebut."},
    {"instruction": "Berdasarkan Pasal 2 Desa Biru, apa yang dimaksud dengan kewenangan lokal berskala desa?", "output": "Berdasarkan Pasal 2 Desa Biru, yang dimaksud dengan kewenangan lokal berskala desa adalah kewenangan yang berlaku secara khusus di wilayah desa Biru dan diatur dalam Peraturan Desa ini."},
    {"instruction": "Berdasarkan Pasal 2 Desa Biru, apakah kewenangan desa Biru termasuk kewenangan berdasarkan hak asal usul atau kewenangan lokal berskala desa?", "output": "Berdasarkan Pasal 2 Desa Biru, kewenangan desa Biru meliputi kewenangan berdasarkan hak asal usul dan kewenangan lokal berskala desa."},
    {"instruction": "Berdasarkan Pasal 2 Desa Biru, apakah Peraturan Desa ini hanya mengatur kewenangan berdasarkan hak asal usul?", "output": "Berdasarkan Pasal 2 Desa Biru, Peraturan Desa ini mengatur kewenangan berdasarkan hak asal usul dan juga kewenangan lokal berskala desa."},
    {"instruction": "Apakah yang dimaksud dengan daftar kewenangan Desa Biru?", "output": "Berdasarkan Pasal 4 Desa Biru, daftar kewenangan Desa Biru berdasarkan hak asal usul sebagaimana dimaksud dalam pasal ini."},
    {"instruction": "Apa yang dimaksud dengan hak asal usul dalam konteks kewenangan Desa Biru?", "output": "Berdasarkan Pasal 4 Desa Biru, hak asal usul merujuk pada sumber-sumber hukum yang memberikan kewenangan kepada Desa Biru."},
    {"instruction": "Apakah kewenangan Desa Biru ditentukan secara spesifik dalam pasal ini?", "output": "Berdasarkan Pasal 4 Desa Biru, kewenangan Desa Biru ditentukan berdasarkan hak asal usul sebagaimana dimaksud dalam pasal ini, yang artinya kewenangan tersebut spesifik dan ditentukan berdasarkan sumber hukum yang relevan."},
    {"instruction": "Apakah ada contoh kewenangan yang spesifik yang disebutkan dalam pasal ini?", "output": "Berdasarkan Pasal 4 Desa Biru, pasal ini tidak menyebutkan contoh kewenangan yang spesifik, melainkan menyatakan bahwa kewenangan Desa Biru ditentukan berdasarkan hak asal usul."},
    {"instruction": "Apakah pasal ini memberikan detail tentang bagaimana menentukan kewenangan Desa Biru?", "output": "Berdasarkan Pasal 4 Desa Biru, pasal ini hanya menyatakan bahwa kewenangan Desa Biru ditentukan berdasarkan hak asal usul, tanpa memberikan detail lebih lanjut tentang bagaimana menentukan kewenangan tersebut."}
]


# Jika Anda punya file JSON/CSV:
# import pandas as pd
# df = pd.read_json("legal_qa.json")
# data = [{"instruction": row["question"], "output": row["answer"]} for _, row in df.iterrows()]

dataset = Dataset.from_list(data)
print(f"Jumlah sampel: {len(dataset)}")

Jumlah sampel: 15


In [5]:
# Cell 4: Tokenisasi dengan template chat Llama 3.1
EOS_TOKEN = tokenizer.eos_token  # Harus ditambahkan di akhir output

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instr, out in zip(instructions, outputs):
        # Template chat resmi Llama 3.1 untuk single-turn
        text = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{instr}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{out}{EOS_TOKEN}"""
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Contoh data setelah diformat:")
print(dataset[0]["text"][:300])

Map: 100%|██████████| 15/15 [00:00<00:00, 6548.14 examples/s]

Contoh data setelah diformat:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintah Pusat adalah siapa?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Berdasarkan Pasal 1 Desa Biru, yang dimaksud dengan Pemerintah Pusat adalah Presiden Republik Indonesi


In [6]:
# Cell 5: Siapkan Trainer dengan SFTTrainer dari Unsloth
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Bisa True untuk efisiensi, False lebih stabil
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 15,          # Sesuaikan dengan jumlah data
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",      # Matikan Wandb/TensorBoard jika tidak perlu
    ),
)

# Mulai training
trainer_stats = trainer.train()
print("Fine-tuning selesai!")

num_proc must be <= 15. Reducing num_proc to 15 for dataset of size 15.
[datasets.arrow_dataset|WARNING]num_proc must be <= 15. Reducing num_proc to 15 for dataset of size 15.
Unsloth: Tokenizing ["text"] (num_proc=15): 100%|██████████| 15/15 [00:02<00:00,  7.06 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15 | Num Epochs = 8 | Total steps = 15
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.681500
2,1.993600
3,1.715400
4,1.696800
5,1.389800
6,1.305000
7,1.003900
8,0.800200
9,0.680600
10,0.656700


Fine-tuning selesai!


In [ ]:
# Cell 6: Simpan adaptor LoRA (hanya beberapa MB)
model.save_pretrained("lora_adapter_legal")       # Folder adaptor
tokenizer.save_pretrained("lora_adapter_legal")
print("Adaptor LoRA disimpan di folder 'lora_adapter_legal'.")



Adaptor LoRA disimpan di folder 'lora_adapter_legal'.
Found HuggingFace hub cache directory: /home/kel7/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [11:17<00:00, 169.45s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:32<00:00,  8.05s/it]


Unsloth: Merge process complete. Saved to `/home/kel7/notebooks/model_merged_legal`
Model gabungan disimpan di 'model_merged_legal'.


In [ ]:
# Cell 7: Uji model setelah fine-tuning
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # Aktifkan mode inferensi

# Siapkan prompt
messages = [{"role": "user", "content": "apa yang dimaksud dengan Kewenangan?"}]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Generate jawaban
outputs = model.generate(
    input_ids,
    max_new_tokens = 512,
    temperature = 0.1,
    do_sample = True,
)
response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print("Respons model:", response)

In [ ]:
# Opsional: gabungkan adaptor ke model dasar (menjadi model penuh)
model.save_pretrained_merged("model_merged_legal", tokenizer, save_method = "merged_16bit")
print("Model gabungan disimpan di 'model_merged_legal'.")

# Load Model Gabungan & Tokenizer

Pakai Transformers (standar)

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "model_merged_legal"   # folder hasil merge

# Muat model dalam 16-bit asli (atau bahkan 8-bit kalau perlu hemat RAM)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,   # atau torch.float16
    device_map="auto"             # otomatis letakkan di GPU
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.36it/s]


AttributeError: 'dict' object has no attribute 'model_type'

Pakai Unsloth (untuk kecepatan optimal)

In [4]:
# import unsloth 
# from unsloth import FastLanguageModel
# import torch

# model_path = "model_merged_legal"

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_path,
#     max_seq_length=2048,
#     dtype=torch.bfloat16,
#     load_in_4bit=False,          # Model sudah 16-bit
# )
# # Aktifkan mode inferensi
# FastLanguageModel.for_inference(model)

Prompt & Generate Jawaban

In [ ]:
# Pertanyaan contoh
pertanyaan = "Apa yang dimaksud dengan Pasal 21 Konstitusi India?"

# Buat prompt sesuai format chat
messages = [
    {"role": "user", "content": pertanyaan}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# Generate jawaban
output_ids = model.generate(
    input_ids,
    max_new_tokens=512,
    temperature=0.1,   # rendah supaya jawaban lebih presisi
    do_sample=True,
    repetition_penalty=1.1
)

# Decode hanya bagian respons (tanpa prompt)
response = tokenizer.decode(
    output_ids[0][input_ids.shape[1]:],
    skip_special_tokens=True
)

print("🤖 Jawaban Model Hukum:\n", response)

NameError: name 'tokenizer' is not defined